<a href="https://colab.research.google.com/github/zippyzippy0/miniproject/blob/main/notebooks/assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assessment of Access to Essential Services in Kenya**

Zipporah Mutua

### Abstract


This notebook investigates spatial inequalities in access to education and healthcare across Kenyan counties, using a combination of administrative boundaries, population data, and facility locations. Data were obtained from authoritative sources, including HDX shapefiles for health and education facilities, and population statistics at county and sub-county levels.

The analysis first maps population distribution against the availability of schools and hospitals, highlighting regions with under- or over-supply. Probabilistic and statistical methods, including Pearson correlation, linear regression, and Bayesian modeling, were employed to quantify the relationship between population and facility counts, as well as distances to nearest hospitals. The study identifies priority counties requiring investment based on facilities per capita metrics, showing significant disparities in access.

## Importing Libraries & Cloning Repository


In [ ]:
%%capture
%pip install osmnx hdx-python-api geopandas shapely unidecode fuzzywuzzy


In [ ]:
!git clone https://github.com/zippyzippy0/miniproject.git


In [ ]:
#rm -rf /content/miniproject/fynesse/__pycache__


In [ ]:
import os, subprocess, importlib, sys

def load_repo(repo: str, module: str):
    """
    Clone (or update) a GitHub repo and import a module from it.

    Args:
        repo (str): GitHub repo in the form 'username/repo-name'
        module (str): Python module inside the repo to import
    """
    repo_name = repo.split("/")[-1]

    if not os.path.exists(repo_name):
        print(f" Cloning {repo} ...")
        subprocess.run(["git", "clone", f"https://github.com/{repo}.git"], check=True)
    else:
        print(f"Updating {repo_name} ...")
        subprocess.run(["git", "-C", repo_name, "pull"], check=True)

    if repo_name not in sys.path:
        sys.path.insert(0, repo_name)

    mod = importlib.import_module(module)
    importlib.reload(mod)
    return mod


In [ ]:
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
sys.path.append("/content/miniproject")
import fynesse


In [ ]:
fynesse = load_repo("zippyzippy0/miniproject", "fynesse")

In [ ]:
from fynesse import access, assess, address


In [ ]:
data_path = os.path.join("miniproject", "data")


In [ ]:
from fynesse.access import (
    load_local_csv,
    load_osm_data,
    load_local_shapefile,
    load_from_github,
    load_file,
    load_osm,
    load_shapefile_from_github,
    download_file,
    init_hdx,
    search_hdx,
    download_hdx_resource
)
from fynesse.assess import (
    bernoulli_access,
    pearson_correlation,
    merge_facilities,
    plot_correlation,
    plot_normalized_stacked,
    compute_distances,
    plot_distance_distribution

)


## **ACCESS**

### Data Sources & Data Readiness Levels (DRLs)





**Datasets**:
- **OpenStreetMap (OSM)** — amenity=school and amenity=hospital/clinic points. *Crowd-sourced: completeness varies; expect undercoverage in rural areas.*
- **HOT OSM / HOTOSM shapefiles** — curated extracts for Kenya (where available). *Useful to improve coverage in some localities.*
- **Kenya administrative boundaries** — national, county and subcounty shapefiles (from KNBS / provided repo). *Generally authoritative for spatial joins.*
- **Kenya census / population rasters or CSVs** — used to compute per-capita facility rates.

###  Loading the Dataset



County, sub-county, and national shapefiles are loaded to provide administrative boundaries.
Health and education facility shapefiles are loaded to provide point-level locations.
Population CSV files are loaded to compute per-capita metrics.
Initial plots confirm data integrity and spatial alignment.

In [ ]:
kenya = load_local_shapefile(os.path.join(data_path, "ken_admbnda_adm0_iebc_20191031.shp"))
counties = load_local_shapefile(os.path.join(data_path, "ke_county.shp"))
subcounties = load_local_shapefile(os.path.join(data_path, "ken_admbnda_adm2_iebc_20191031.shp"))

health = load_local_shapefile(os.path.join(data_path, "hotosm_ken_health_facilities_points_shp/hotosm_ken_health_facilities_points_shp.shp"))
schools = load_local_shapefile(os.path.join(data_path, "hotosm_ken_education_facilities_points_shp/hotosm_ken_education_facilities_points_shp.shp"))

population_county = load_local_csv(os.path.join(data_path, "kenya-population-by-sex-and-county.csv"))
population_subcounty = load_local_csv(os.path.join(data_path, "kenya-population-by-sub-county.csv"))


In [ ]:
kenya.plot(figsize=(10,8))


In [ ]:
counties.plot(figsize=(10,8))

In [ ]:
subcounties.plot(figsize=(10,8))


### Exploratory Check of Facilities




The number of health facilities and schools is printed. The first rows of each dataset are displayed.
This ensures that data is correctly loaded and ready for spatial joins.

In [ ]:
print(len(health))
print(len(schools))

In [ ]:
counties.head()

In [ ]:
health.head()

In [ ]:
schools.head()

In [ ]:
kenya.head()

In [ ]:
population_county.head()

In [ ]:
population_county = population_county[population_county["name"] != "Kenya"]


In [ ]:
population_county.head()

In [ ]:
def clean_population_county(df, county_col="name", pop_col="Total"):
    df[county_col] = df[county_col].str.strip()
    corrections = {
        "Taita-Taveta": "Taita Taveta",
        "Tharaka-Nithi": "Tharaka-nithi",
        "Elgeyo-Marakwet": "Elgeyo-marakwet",
        "Murang'A": "Murang'a"
    }
    df[county_col] = df[county_col].replace(corrections)
    return df

In [ ]:
population_county.info()

In [ ]:
population_subcounty.head()

In [ ]:
counties[counties['county'].isna()]


In [ ]:
counties = counties.rename(columns={"county": "ADM1_EN"})
print(counties.head())


In [ ]:
population_county[population_county['Total'].isna()]


In [ ]:
gdf = counties.merge(population_county, left_on="ADM1_EN", right_on="name", how="left")
gdf[['ADM1_EN', 'Total']].head()

## **ASSESS**

### Probabilistic & Statistical Analysis





**Purpose.**



This section performs the statistical and probabilistic analyses that quantify access inequalities. It complements the descriptive maps and counts with probabilistic statements and uncertainty estimates.

### Joining Facilities to Counties




Facility points are transformed to match the county CRS.
Spatial joins attach health and school points to counties.
Aggregated counts of facilities per county are computed for later normalization and comparison.

In [ ]:
points_health = health.to_crs(counties.crs)
points_schools = schools.to_crs(counties.crs)

# Spatial joins
joined_health = gpd.sjoin(points_health, counties, how="right", predicate="within")
joined_schools = gpd.sjoin(points_schools, counties, how="right", predicate="within")

counts_health = joined_health.groupby("ADM1_EN").size().reset_index(name="hospitals")
counts_schools = joined_schools.groupby("ADM1_EN").size().reset_index(name="schools")

In [ ]:
all_counties = gpd.GeoDataFrame({'ADM1_EN': list(counties['ADM1_EN'].unique())})
counts_health = all_counties.merge(counts_health, on='ADM1_EN', how='left').fillna(0)


### Population Counts

Population data is aggregated by county and renamed to match administrative column names.
This enables merging population data with facility counts for per-capita calculations.


In [ ]:
pop_counts = population_county.groupby("name", as_index=False)["Total"].sum()
pop_counts.rename(columns={"name": "ADM1_EN", "Total": "pop_total"}, inplace=True)


In [ ]:
print(len(points_schools), "points in total")
print(len(joined_schools), "points joined to counties")

print(joined_schools["ADM1_EN"].nunique(), "unique counties in joined data")
print(joined_schools["ADM1_EN"].value_counts().tail(10))

In [ ]:
print(len(points_health), "points in total")
print(len(joined_health), "points joined to counties")

# Check distinct counties matched
print(joined_health["ADM1_EN"].nunique(), "unique counties in joined data")
print(joined_health["ADM1_EN"].value_counts().tail(10))

### Facility-Population Merge


Health and school counts are merged with population data to form a consolidated DataFrame.
Columns are normalized as percentages to allow comparison across counties.


Merging

In [ ]:
df = assess.merge_facilities(pop_counts, counts_schools, counts_health)

In [ ]:
df.head()

Normalization

In [ ]:
cols_to_norm = ["pop_total", "schools", "hospitals"]
df[[f"{c}_norm" for c in cols_to_norm]] = df[cols_to_norm].apply(lambda x: x / x.sum() * 100)


### OSM Data Fetch

An Overpass API function is defined to fetch schools and hospitals using bounding boxes.


In [ ]:
import requests
import geopandas as gpd
from shapely.geometry import Point

def fetch_osm_amenities(bbox, amenities):
    """Fetches OSM amenities within a bbox using list comprehension."""
    query = f"""
    [out:json][timeout:180];
    nwr["amenity"~"{'|'.join(amenities)}"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
    out center;
    """
    url = "http://overpass-api.de/api/interpreter"
    try:
        response = requests.get(url, params={"data": query}, timeout=180)
        response.raise_for_status()
        data = response.json().get("elements", [])

        features = [
            {
                "name": el["tags"].get("name", "Unknown"),
                "amenity": el["tags"].get("amenity", "Unknown"),
                "geometry": Point(el.get("lon", el.get("center", {}).get("lon")), el.get("lat", el.get("center", {}).get("lat")))
            } for el in data if "lon" in el or "center" in el
        ]

        return gpd.GeoDataFrame(features, crs="EPSG:4326")

    except requests.exceptions.RequestException as e:
        print(f"Overpass API request failed: {e}")
        return gpd.GeoDataFrame(columns=["name", "amenity", "geometry"], geometry="geometry", crs="EPSG:4326")


In [ ]:
kenya_bbox = [-4.72, 33.89, 5.33, 41.89]
points = fetch_osm_amenities(kenya_bbox, ["school", "hospital"])

kenya_union = counties.unary_union
points = points[points.within(kenya_union)]

### Visualizing Facilities Using OSM

County-level maps visualize OSM-fetched schools and hospitals.
The plots reveal clustering in urban areas and gaps in rural counties, reinforcing the need for more complete datasets.


In [ ]:
fig, ax = plt.subplots(figsize=(12,10))
counties.boundary.plot(ax=ax, color="black", linewidth=0.5)
points[points["amenity"]=="school"].plot(ax=ax, color="blue", markersize=5, alpha=0.5, label="Schools")
points[points["amenity"]=="hospital"].plot(ax=ax, color="red", markersize=5, alpha=0.7, label="Hospitals")
plt.legend()
plt.title("Schools and Hospitals inside Kenya (OSM)")
plt.show()

A plot to show hospital and school distributions in Kitui County




In [ ]:
county_name = "Kitui"
county = counties[counties["ADM1_EN"] == county_name]
bbox = county.total_bounds
points = fetch_osm_amenities([bbox[1], bbox[0], bbox[3], bbox[2]], ["school", "hospital"])

county_union = county.unary_union
points = points[points.within(county_union)]

fig, ax = plt.subplots(figsize=(10,8))
county.boundary.plot(ax=ax, color="black", linewidth=1)
points[points["amenity"]=="school"].plot(ax=ax, color="blue", markersize=8, alpha=0.5, label="Schools")
points[points["amenity"]=="hospital"].plot(ax=ax, color="red", markersize=8, alpha=0.7, label="Hospitals")
plt.legend()
plt.title(f"Schools and Hospitals in {county_name}")
plt.show()

A plot to show hospital and school distributions in Nairobi County


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

county_name = "Nairobi"
county = counties[counties["ADM1_EN"] == county_name]

if not county.empty:
    minx, miny, maxx, maxy = county.geometry.iloc[0].bounds

    bbox = [miny, minx, maxy, maxx]

    points = fetch_osm_amenities(bbox, ["school", "hospital"])
    if not points.empty:
        county_union = county.geometry.union_all()
        points = points[points.within(county_union)]

        fig, ax = plt.subplots(figsize=(10,8))
        county.boundary.plot(ax=ax, color="black", linewidth=1)
        points[points["amenity"]=="school"].plot(ax=ax, color="blue", markersize=8, alpha=0.5, label="Schools")
        points[points["amenity"]=="hospital"].plot(ax=ax, color="red", markersize=8, alpha=0.7, label="Hospitals")
        plt.legend()
        plt.title(f"Schools and Hospitals in {county_name}")
        plt.show()
    else:
        print(f"No amenities data fetched for {county_name}.")
else:
    print(f"County '{county_name}' not found.")

### Humanitarian Data Visualization

Having done facility distribution using OSM, the distribution apppers unevenly distributed and therefore gives inaccurate and incomplete coverage.

As a result dataset from Human Data Exchange is used tu overcome the OSM shortcomings.
 HDX health and school facility datasets are plotted over county boundaries.
Separate maps show distribution per type of facility, while combined maps show overall coverage.


Facility Counts per County



Bar plots display the number of health facilities and schools per county.
This enables quick identification of counties with high or low facility counts.


In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
counties.plot(ax=ax, color="lightgrey", edgecolor="black")
health.plot(ax=ax, color="red", markersize=5)
ax.set_title("Health Facilities")

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
counties.plot(ax=ax, color="lightgrey", edgecolor="black")
schools.plot(ax=ax, color="yellow", markersize=5)
ax.set_title("Health Facilities")

### Choropleth Maps of Facilities
Interactive choropleth maps show spatial distribution of health facilities, schools, and combined counts.
These maps provide a clear geographic perspective on service coverage.


In [ ]:
combined = pd.merge(counts_health, counts_schools, on="ADM1_EN", how="outer").fillna(0)
combined["total"] = combined["hospitals"] + combined["schools"]

In [ ]:
gdf_combined = combined.merge(counties[['ADM1_EN','geometry']], on="ADM1_EN").set_geometry("geometry")


In [ ]:
import plotly.express as px

# Plot combined choropleth
fig = px.choropleth_mapbox(
    gdf_combined,
    geojson=gdf_combined.__geo_interface__,
    locations="ADM1_EN",
    color="total",
    featureidkey="properties.ADM1_EN",
    hover_name="ADM1_EN",
    hover_data=["hospitals", "schools", "total"],
    mapbox_style="carto-positron",
    zoom=5,
    center={"lat":0.1, "lon":37.9},
    color_continuous_scale="Viridis"
)

fig.show()

### Population vs Facilities Stacked Plots


Population and school counts are visualized using stacked bar plots.
Logarithmic scaling highlights disparities in facility allocation relative to population size.


In [ ]:
import plotly.express as px
df_sorted = df.sort_values("pop_total_norm")
df_sorted["pop_total"] = df_sorted["pop_total_norm"] + 1e-3
df_sorted["schools"] = df_sorted["schools_norm"] + 1e-3

fig = px.bar(
    df_sorted,
    y="ADM1_EN",
    x=["pop_total", "schools"],
    orientation='h',
    barmode='stack',
    labels={"value": "Percentage Share (%)", "ADM1_EN": "County"},
    height=2000
)

fig.update_layout(
    title="Population vs Schools per County",
    xaxis_type="log"
)

fig.show()


#### Correlation between Population, Hospitals and Schools



In [ ]:
df["schools_per_100k"] = (df["schools"] / df["pop_total"]) * 100000
df["hospitals_per_100k"] = (df["hospitals"] / df["pop_total"]) * 100000


In [ ]:
corr_percapita = df[["schools_per_100k", "hospitals_per_100k"]].corr()
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(corr_percapita, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation between Per-Capita Schools and Hospitals per County")
plt.show()


In [ ]:
df_sorted = df.sort_values("schools_per_100k", ascending=False)
fig = px.bar(
    df_sorted,
    x="schools_per_100k",
    y="ADM1_EN",
    orientation="h",
    title="Schools per 100,000 People by County",
    labels={"ADM1_EN": "County", "schools_per_100k": "Schools per 100k"}
)
fig.show()


In [ ]:
df_sorted = df.sort_values("hospitals_per_100k", ascending=False)
fig = px.bar(
    df_sorted,
    x="hospitals_per_100k",
    y="ADM1_EN",
    orientation="h",
    title="Hospitals per 100,000 People by County",
    labels={"ADM1_EN": "County", "hospitals_per_100k": "Hospitals per 100k"}
)
fig.show()


####  Plots & Regression for Hospitals



Regression plots visualize relationships between population and number of
 hospitals per county.
Trends and outliers are identified visually, supporting correlation analyses.


In [ ]:
df_no_nairobi = df.loc[df["ADM1_EN"] != "Nairobi"].copy()
df_no_nairobi["hospitals_per_capita"] = df_no_nairobi["hospitals"] / df_no_nairobi["pop_total"]
df_no_nairobi["schools_per_capita"] = df_no_nairobi["schools"] / df_no_nairobi["pop_total"]


In [ ]:
sns.regplot(
    data=df_no_nairobi,
    x="pop_total",
    y="hospitals",
    scatter_kws={"alpha":0.6},
    line_kws={"color":"red"}
)
plt.xlabel("Population")
plt.ylabel("Number of Hospitals")
plt.title("Population vs. Hospitals per County")
plt.show()


In [ ]:
sns.regplot(
    data=df_no_nairobi,
    x="pop_total",
    y="schools",
    scatter_kws={"alpha":0.6},
    line_kws={"color":"red"}
)
plt.xlabel("Population")
plt.ylabel("Number of schoolss")
plt.title("Population vs. schools per County")
plt.show()


### Bayesian Regression Models



Bayesian linear regression is applied to estimate the relationship between population and facility counts.
Posterior distributions provide estimates for intercept, slope, and uncertainty.
This approach complements traditional linear regression with probabilistic inference.

In [ ]:
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# Standardize predictor
X = (df["pop_total"] - df["pop_total"].mean()) / df["pop_total"].std()
y_schools = df["schools"]
y_health = df["hospitals"]

In [ ]:
def run_bayesian_regression(y, label):
    with pm.Model() as model:
        alpha = pm.Normal(f"alpha_{label}", mu=0, sigma=10)
        beta = pm.Normal(f"beta_{label}", mu=0, sigma=10)
        sigma = pm.HalfNormal(f"sigma_{label}", sigma=1)
        mu = alpha + beta * X
        y_obs = pm.Normal(f"y_obs_{label}", mu=mu, sigma=sigma, observed=y)
        trace = pm.sample(1000, tune=1000, target_accept=0.9, chains=2, cores=1)
    return trace

In [ ]:
trace_schools = run_bayesian_regression(y_schools, "schools")


In [ ]:
az.plot_posterior(trace_schools, var_names=["alpha_schools","beta_schools"])
plt.show()

az.summary(trace_schools, var_names=["alpha_schools","beta_schools","sigma_schools"])


In [ ]:
trace_health = run_bayesian_regression(y_health, "health")


In [ ]:
az.plot_posterior(trace_health, var_names=["alpha_health","beta_health"])
plt.show()

az.summary(trace_health, var_names=["alpha_health","beta_health","sigma_health"])


### Distance to Nearest Hospital


Distances from county centroids to nearest hospitals are computed using projected coordinates.
Histograms visualize distance distributions, and probabilities of access within a threshold are calculated.
This quantifies accessibility in practical terms.

In [ ]:
from shapely.ops import nearest_points
from scipy.stats import norm
import numpy as np
counties_projected = counties.to_crs(epsg=32737)
health_projected = health.to_crs(epsg=32737)
distances_km = []

for county_name in df["ADM1_EN"]:
    county_geom = counties_projected[counties_projected["ADM1_EN"]==county_name].geometry.iloc[0]
    centroid = county_geom.centroid
    nearest_distance = health_projected.distance(centroid).min()
    distances_km.append(nearest_distance / 1000)

df["dist_to_nearest_hospital_km"] = distances_km
mu, std = norm.fit(df["dist_to_nearest_hospital_km"])
sns.histplot(df["dist_to_nearest_hospital_km"], kde=True, color="green")
plt.axvline(mu, color="red", linestyle="--")
plt.title(f"Distances to Nearest Hospital (mean={mu:.2f} km)")
plt.show()

In [ ]:
import numpy as np
import pandas as pd

threshold_km = 17
bernoulli_outcome = (df["dist_to_nearest_hospital_km"] <= threshold_km).astype(int)

prob_access = bernoulli_outcome.mean()
print(f"Probability of having a facility within {threshold_km} km: {prob_access:.2%}")

## ADDRESS


### Policy Recommendations & Prioritisation





**Purpose.** Translate analysis into clear recommendations for policymakers and stakeholders.

Top counties needing schools or hospitals are displayed using color-coded tables and bar plots.
These visualizations support decision-making for resource allocation.


In [ ]:
from fynesse.address import rank_underserved_regions, suggest_priority_areas


#### Top 5 counties that need schools/hospitals

In [ ]:
underserved_schools, underserved_hospitals = address.get_underserved(df, top_n=5)

print("Priority counties for school investment:")
display(underserved_schools.style.background_gradient(cmap="Oranges"))

print("\nPriority counties for hospital investment:")
display(underserved_hospitals.style.background_gradient(cmap="Reds"))


### Priority Counties for Investment

These plots highlight counties with the lowest schools and hospitals per capita. They provide a clear, visual basis for prioritizing resource allocation. Counties at the top of each chart are the most underserved and should be targeted first for development interventions. This aligns with the analysis showing that population alone does not fully explain facility distribution, emphasizing the need for per-capita planning.

###  Predictive Linear Regression


Linear regression models predict expected facility counts given population size.
R² scores measure how much variance in facilities is explained by population.
Predicted versus actual plots provide visual validation of model performance.


####  Model Evaluation


The trained model is evaluated here to assess its performance. Even if prediction accuracy is low, documenting and analyzing results is important.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
X_s = df_no_nairobi[["pop_total"]]
y_s = df_no_nairobi["schools"]

In [ ]:
model_s = LinearRegression()
model_s.fit(X_s, y_s)

In [ ]:
r2_s = r2_score(y_s, model_s.predict(X_s))


In [ ]:
X_h = df_no_nairobi[["pop_total"]]
y_h = df_no_nairobi["hospitals"]

In [ ]:
model_h = LinearRegression()
model_h.fit(X_h, y_h)

In [ ]:
r2_h = r2_score(y_h, model_h.predict(X_h))

In [ ]:
r2_s, r2_h

In [ ]:
pop_val = 1_500_000
pred_schools = model_s.predict(pd.DataFrame({"pop_total":[pop_val]}))[0]
pred_hospitals = model_h.predict(pd.DataFrame({"pop_total":[pop_val]}))[0]

print(f"Predicted schools for population {pop_val:,}: {pred_schools:.0f}")
print(f"Predicted hospitals for population {pop_val:,}: {pred_hospitals:.0f}")


This means that, according to the model, a county of this population(1_500_000) would ideally have around 427 schools and 119 hospitals to maintain proportional access.

This prediction helps identify counties that are under- or over-served relative to their population.


#### Predicting Schools/Hospitals per County


This section visualizes the relationship between county population and the number of schools or hospitals.  
A linear regression model is used to predict the number of schools/hospitals based on population, and the predicted values are plotted against actual data.  
The red line represents the model fit, and the R² value indicates how well population explains the variation in school/hospital counts.


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=df_no_nairobi["pop_total"], y=df_no_nairobi["hospitals"], alpha=0.7)

# Range for the line
x_min, x_max = df_no_nairobi["pop_total"].min(), df_no_nairobi["pop_total"].max()
x_range = np.linspace(x_min, x_max, 100)
x_range_df = pd.DataFrame(x_range, columns=["pop_total"])
y_pred_range = model_h.predict(x_range_df)

plt.plot(x_range, y_pred_range, color="red", label=f"R² = {r2_h:.2f}")
plt.xlabel("Population")
plt.ylabel("Number of Health Facilities")
plt.title("Population vs Health Facilities (per County)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=df_no_nairobi["pop_total"], y=df_no_nairobi["schools"], alpha=0.7)
x_min, x_max = df_no_nairobi["pop_total"].min(), df_no_nairobi["pop_total"].max()
x_range = np.linspace(x_min, x_max, 100)
x_range_df = pd.DataFrame(x_range, columns=["pop_total"])
y_pred_range = model_s.predict(x_range_df)

plt.plot(x_range, y_pred_range, color="red", label=f"R² = {r2_s:.2f}")
plt.xlabel("Population")
plt.ylabel("Number of Schools")
plt.title("Population vs Schools per County")
plt.legend()
plt.show()

### Interactive Visualization



A scatter plot shows population, schools, and hospitals in an interactive view.
This allows exploration of county-level data and supports evidence-based recommendations.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

In [ ]:
df_plot = df.copy()
for col in ["pop_total", "schools", "hospitals"]:
    df_plot[col] = df_plot[col].clip(lower=1)
df_plot["schools_pc"] = df_plot["schools"] / df_plot["pop_total"]
df_plot["hospitals_pc"] = df_plot["hospitals"] / df_plot["pop_total"]

df_plot["log_pop"] = np.log(df_plot["pop_total"])
df_plot["log_schools"] = np.log(df_plot["schools"])
df_plot["log_hospitals"] = np.log(df_plot["hospitals"])

def style_full(fig, title):
    fig.update_layout(
        template="plotly_white",
        autosize=True,
        width=None,
        height=700,
        margin=dict(l=30, r=30, t=70, b=40),
        title=title,
        legend_title="County"
    )
    return fig


In [ ]:
fig1 = px.scatter(
    df_plot,
    x="hospitals",
    y="schools",
    size="pop_total",
    color="ADM1_EN",
    hover_name="ADM1_EN",
    title="PLOT 1: Schools vs Hospitals (Log–Log Scale)",
    size_max=60
)
fig1.update_xaxes(type="log")
fig1.update_yaxes(type="log")
fig1 = style_full(fig1, fig1.layout.title.text)

fig1.show()


In [ ]:
fig2 = px.scatter(
    df_plot,
    x="hospitals_pc",
    y="schools_pc",
    size="pop_total",
    color="ADM1_EN",
    hover_name="ADM1_EN",
    title="PLOT 2: Schools vs Hospitals per Capita",
    size_max=60
)
fig2.update_layout(
    xaxis_title="Hospitals per Person",
    yaxis_title="Schools per Person"
)
fig2 = style_full(fig2, fig2.layout.title.text)

fig2.show()


In [ ]:
fig3 = px.scatter(
    df_plot,
    x="pop_total",
    y="schools",
    color="ADM1_EN",
    size="pop_total",
    trendline="ols",
    title="PLOT 3: Schools vs Population (Log–Log with Trendline)",
    size_max=50
)
fig3.update_xaxes(type="log")
fig3.update_yaxes(type="log")
fig3 = style_full(fig3, fig3.layout.title.text)
fig3.show()


In [ ]:
fig4 = px.scatter(
    df_plot,
    x="pop_total",
    y="hospitals",
    color="ADM1_EN",
    size="pop_total",
    trendline="ols",
    title="PLOT 4: Hospitals vs Population (Log–Log with Trendline)",
    size_max=50
)
fig4.update_xaxes(type="log")
fig4.update_yaxes(type="log")
fig4 = style_full(fig4, fig4.layout.title.text)

fig4.show()
